# Transaction data quality

This notebook analyses data quality problems in a deliberately corrupted copy of the transactions dataset

## Objectives

- Create a reproducible dataset containing known data quality problems
- Detect missing values
- Identify duplicate rows and duplicate transaction identifiers
- Detect invalid numeric values and dates
- Find inconsistent categorical values
- Validate account and merchant references
- Prepare the dataset for cleaning

In [1]:
from pathlib import Path
import pandas as pd

In [2]:
current_directory = Path.cwd()

if current_directory.name == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory

project_root

WindowsPath('c:/Users/Focus/Documents/GitHub/transaction-intelligence')

In [3]:
raw_transactions_path = (project_root / "data" / "raw" / "transactions.csv")
dirty_data_directory = (project_root / "data" / "dirty")
dirty_transactions_path = (dirty_data_directory / "transactions.csv")

print(f"Raw file: {raw_transactions_path}")
print(f"Dirty directory: {dirty_data_directory}")
print(f"Dirty file: {dirty_transactions_path}")

Raw file: c:\Users\Focus\Documents\GitHub\transaction-intelligence\data\raw\transactions.csv
Dirty directory: c:\Users\Focus\Documents\GitHub\transaction-intelligence\data\dirty
Dirty file: c:\Users\Focus\Documents\GitHub\transaction-intelligence\data\dirty\transactions.csv


In [4]:
dirty_data_directory.mkdir(parents = True, exist_ok = True)

In [5]:
dirty_data_directory.exists()

True

## Loading original dataset

The original transactions file is preserved unchanged; I create a separate copy before introducing any data quality problems

In [6]:
clean_transactions = pd.read_csv(raw_transactions_path)
clean_transactions.head()

,transaction_id,account_id,merchant_id,timestamp,amount,currency,transaction_type,status
0,T0001,A001,M001,2025-01-03 08:15:00,34.80,EUR,Purchase,Completed
1,T0002,A003,M002,2025-01-04 10:42:00,4.20,EUR,Purchase,Completed
2,T0003,A005,M003,2025-01-05 19:10:00,18.99,EUR,Purchase,Completed
3,T0004,A006,M004,2025-01-06 21:35:00,249.90,EUR,Purchase,Completed
4,T0005,A008,M005,2025-01-08 12:05:00,62.40,EUR,Purchase,Completed


In [7]:
clean_transactions.shape

(60, 8)

In [8]:
dirty_transactions = clean_transactions.copy(deep = True)

In [9]:
dirty_transactions.shape

(60, 8)

In [10]:
dirty_transactions["amount"] =  (
    dirty_transactions["amount"].astype("string")
)

In [11]:
dirty_transactions.dtypes

transaction_id         str
account_id             str
merchant_id            str
timestamp              str
amount              string
currency               str
transaction_type       str
status                 str
dtype: object

In [12]:
# Here I will introduce some data quality issues to the transactions dataset

dirty_transactions.loc[dirty_transactions["transaction_id"] == "T0007", "amount"] = pd.NA
dirty_transactions.loc[dirty_transactions["transaction_id"] == "T0028", "currency"] = pd.NA
dirty_transactions.loc[dirty_transactions["transaction_id"] == "T0018", "amount"] = "not_available"
dirty_transactions.loc[dirty_transactions["transaction_id"] == "T0040", "amount"] = "-68.20"
dirty_transactions.loc[dirty_transactions["transaction_id"] == "T0012", "timestamp"] = "2025-02-30 09:45:00"
dirty_transactions.loc[dirty_transactions["transaction_id"] == "T0032", "timestamp"] = "14/02/2025 21:10"
dirty_transactions.loc[dirty_transactions["transaction_id"] == "T0014", "currency"] = "usd"
dirty_transactions.loc[dirty_transactions["transaction_id"] == "T0039", "transaction_type"] = "purchase"
dirty_transactions.loc[dirty_transactions["transaction_id"] == "T0035", "status"] = "declined "
dirty_transactions.loc[dirty_transactions["transaction_id"] == "T0057", "status"] = "Complete"
dirty_transactions.loc[dirty_transactions["transaction_id"] == "T0051","account_id"] = "A999"
dirty_transactions.loc[dirty_transactions["transaction_id"] == "T0054","merchant_id"] = "M999"

exact_duplicate = dirty_transactions.loc[dirty_transactions["transaction_id"] == "T0022"].copy()
dirty_transactions = pd.concat([dirty_transactions, exact_duplicate], ignore_index= True)


In [13]:
dirty_transactions.loc[dirty_transactions["transaction_id"].isin(["T0007", "T0028","T0018", "T0040"])]

,transaction_id,account_id,merchant_id,timestamp,amount,currency,transaction_type,status
6,T0007,A011,M007,2025-01-10 18:50:00,<NA>,EUR,Purchase,Completed
17,T0018,A003,M004,2025-01-26 12:44:00,not_available,EUR,Purchase,Completed
27,T0028,A002,M014,2025-02-09 14:30:00,9.8,NaN,Purchase,Completed
39,T0040,A014,M005,2025-02-27 12:10:00,-68.20,EUR,Purchase,Completed


In [14]:
dirty_transactions.shape

(61, 8)

In [15]:
duplicate_identifier = dirty_transactions.loc[dirty_transactions["transaction_id"] == "T0045"].copy()
duplicate_identifier["merchant_id"] = "M010"
duplicate_identifier["amount"] = "99.99"
dirty_transactions = pd.concat([dirty_transactions, duplicate_identifier], ignore_index= True)

In [16]:
dirty_transactions.shape

(62, 8)

In [17]:
# Save corrupted dataset
dirty_transactions.to_csv(dirty_transactions_path, index = False)

In [18]:
dirty_transactions_path.exists()

True

In [19]:
transactions_dirty = pd.read_csv(dirty_transactions_path)

In [20]:
transactions_dirty.head()

,transaction_id,account_id,merchant_id,timestamp,amount,currency,transaction_type,status
0,T0001,A001,M001,2025-01-03 08:15:00,34.8,EUR,Purchase,Completed
1,T0002,A003,M002,2025-01-04 10:42:00,4.2,EUR,Purchase,Completed
2,T0003,A005,M003,2025-01-05 19:10:00,18.99,EUR,Purchase,Completed
3,T0004,A006,M004,2025-01-06 21:35:00,249.9,EUR,Purchase,Completed
4,T0005,A008,M005,2025-01-08 12:05:00,62.4,EUR,Purchase,Completed


In [21]:
transactions_dirty.shape

(62, 8)

In [22]:
transactions_dirty.dtypes

transaction_id      str
account_id          str
merchant_id         str
timestamp           str
amount              str
currency            str
transaction_type    str
status              str
dtype: object

# Initial data quality audit

I will now inspect the corrupted dataset without modifying or cleaning it

The goal is to identify and quantify each type of problem before deciding how it should be handled

In [23]:
transactions_dirty.info()

<class 'pandas.DataFrame'>
RangeIndex: 62 entries, 0 to 61
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   transaction_id    62 non-null     str  
 1   account_id        62 non-null     str  
 2   merchant_id       62 non-null     str  
 3   timestamp         62 non-null     str  
 4   amount            61 non-null     str  
 5   currency          61 non-null     str  
 6   transaction_type  62 non-null     str  
 7   status            62 non-null     str  
dtypes: str(8)
memory usage: 4.0 KB


In [24]:
transactions_dirty.head()

,transaction_id,account_id,merchant_id,timestamp,amount,currency,transaction_type,status
0,T0001,A001,M001,2025-01-03 08:15:00,34.8,EUR,Purchase,Completed
1,T0002,A003,M002,2025-01-04 10:42:00,4.2,EUR,Purchase,Completed
2,T0003,A005,M003,2025-01-05 19:10:00,18.99,EUR,Purchase,Completed
3,T0004,A006,M004,2025-01-06 21:35:00,249.9,EUR,Purchase,Completed
4,T0005,A008,M005,2025-01-08 12:05:00,62.4,EUR,Purchase,Completed


In [25]:
transactions_dirty.tail()

,transaction_id,account_id,merchant_id,timestamp,amount,currency,transaction_type,status
57,T0058,A002,M011,2025-03-27 19:35:00,76.0,EUR,Purchase,Completed
58,T0059,A006,M004,2025-03-29 23:15:00,459.0,EUR,Purchase,Declined
59,T0060,A001,M003,2025-03-31 18:55:00,19.99,EUR,Refund,Completed
60,T0022,A011,M002,2025-02-02 11:25:00,5.6,EUR,Purchase,Completed
61,T0045,A006,M010,2025-03-06 11:05:00,99.99,EUR,Purchase,Completed


In [26]:
# Find missing values
missing_values = (transactions_dirty.isna().sum().sort_values(ascending = False))
missing_values

currency            1
amount              1
transaction_id      0
account_id          0
timestamp           0
merchant_id         0
transaction_type    0
status              0
dtype: int64

In [27]:
transactions_dirty.loc[transactions_dirty.isna().any(axis = 1)]

,transaction_id,account_id,merchant_id,timestamp,amount,currency,transaction_type,status
6,T0007,A011,M007,2025-01-10 18:50:00,NaN,EUR,Purchase,Completed
27,T0028,A002,M014,2025-02-09 14:30:00,9.8,NaN,Purchase,Completed


In [28]:
# Find duplicated rows
transactions_dirty.duplicated().sum()

np.int64(1)

In [29]:
exact_duplicated_rows = transactions_dirty.loc[transactions_dirty.duplicated(keep = False)]
exact_duplicated_rows 

,transaction_id,account_id,merchant_id,timestamp,amount,currency,transaction_type,status
21,T0022,A011,M002,2025-02-02 11:25:00,5.6,EUR,Purchase,Completed
60,T0022,A011,M002,2025-02-02 11:25:00,5.6,EUR,Purchase,Completed


In [30]:
# Find duplicated identificators

duplicated_id_mask = (transactions_dirty["transaction_id"].duplicated(keep = False))

duplicated_transaction_ids = (transactions_dirty.loc[duplicated_id_mask].sort_values("transaction_id"))
duplicated_transaction_ids

,transaction_id,account_id,merchant_id,timestamp,amount,currency,transaction_type,status
21,T0022,A011,M002,2025-02-02 11:25:00,5.6,EUR,Purchase,Completed
60,T0022,A011,M002,2025-02-02 11:25:00,5.6,EUR,Purchase,Completed
44,T0045,A006,M012,2025-03-06 11:05:00,26.4,EUR,Purchase,Completed
61,T0045,A006,M010,2025-03-06 11:05:00,99.99,EUR,Purchase,Completed


In [31]:
duplicated_transaction_ids["transaction_id"].nunique()

2

In [32]:
# Examine categories

transactions_dirty["currency"].value_counts(dropna= False)

currency
EUR    58
USD     2
usd     1
NaN     1
Name: count, dtype: int64

In [33]:
transactions_dirty["status"].value_counts(dropna=False)

status
Completed    58
Declined      2
declined      1
Complete      1
Name: count, dtype: int64

In [34]:
transactions_dirty["transaction_type"].value_counts(dropna = False)

transaction_type
Purchase    60
purchase     1
Refund       1
Name: count, dtype: int64

In [35]:
amount_numeric = pd.to_numeric(transactions_dirty["amount"], errors= "coerce")

invalid_numeric_amounts = transactions_dirty.loc[amount_numeric.isna(), ["transaction_id", "amount"]]
invalid_numeric_amounts

,transaction_id,amount
6,T0007,NaN
17,T0018,not_available


In [36]:
# Find amounts that are not positive

non_positive_amounts = transactions_dirty.loc[amount_numeric.notna() & (amount_numeric <= 0), ["transaction_id", "amount"]]
non_positive_amounts

,transaction_id,amount
39,T0040,-68.20


In [37]:
# Temporal version for datetime conversion

timestamp_parsed = pd.to_datetime(transactions_dirty["timestamp"], format = "mixed", errors = "coerce")

# Exposure of invalid dates

invalid_timestamps = transactions_dirty.loc[timestamp_parsed.isna(), ["transaction_id", "timestamp"]]
invalid_timestamps

,transaction_id,timestamp
11,T0012,2025-02-30 09:45:00


In [38]:
len("2025-02-30 09:45:00")

19

In [39]:
non_standard_timestamps = transactions_dirty.loc[transactions_dirty["timestamp"].str.len().ne(19)]
non_standard_timestamps

,transaction_id,account_id,merchant_id,timestamp,amount,currency,transaction_type,status
31,T0032,A001,M010,14/02/2025 21:10,15.5,EUR,Purchase,Completed


In [40]:
# Now I will validate accounts and merchants datasets

accounts_path = (project_root / "data" / "raw"/ "accounts.csv")

merchants_path = (project_root / "data" / "raw" / "merchants.csv")

accounts = pd.read_csv(accounts_path)
merchants = pd.read_csv(merchants_path)

In [41]:
# Find non existant accounts

invalid_account_references = transactions_dirty.loc[~transactions_dirty["account_id"].isin(accounts["account_id"]),
                                                    ["transaction_id", "account_id"]]

invalid_account_references

,transaction_id,account_id
50,T0051,A999


In [42]:
# Find non existant merchants

invalid_merchant_references = transactions_dirty.loc[~transactions_dirty["merchant_id"].isin(merchants["merchant_id"]),
                                                    ["transaction_id", "merchant_id"]]

invalid_merchant_references

,transaction_id,merchant_id
53,T0054,M999


In [43]:
# Summary

quality_report = pd.Series(
    {
        "number_of_rows": len(
            transactions_dirty
        ),
        "missing_values": int(
            transactions_dirty
            .isna()
            .sum()
            .sum()
        ),
        "exact_duplicate_rows": int(
            transactions_dirty
            .duplicated()
            .sum()
        ),
        "duplicated_transaction_ids": int(
            duplicated_transaction_ids[
                "transaction_id"
            ].nunique()
        ),
        "invalid_or_missing_amounts": int(
            amount_numeric
            .isna()
            .sum()
        ),
        "non_positive_amounts": int(
            len(non_positive_amounts)
        ),
        "invalid_timestamps": int(
            timestamp_parsed
            .isna()
            .sum()
        ),
        "non_standard_timestamps": int(
            len(non_standard_timestamps)
        ),
        "invalid_account_references": int(
            len(invalid_account_references)
        ),
        "invalid_merchant_references": int(
            len(invalid_merchant_references)
        ),
    },
    name="count",
)

quality_report

number_of_rows                 62
missing_values                  2
exact_duplicate_rows            1
duplicated_transaction_ids      2
invalid_or_missing_amounts      2
non_positive_amounts            1
invalid_timestamps              1
non_standard_timestamps         1
invalid_account_references      1
invalid_merchant_references     1
Name: count, dtype: int64

## Initial quality findings

The corrupted transactions dataset contains several deliberately introduced problems:

- The dataset contains 62 rows instead of the original 60
- Two cells contain missing values
- One row is an exact duplicate
- Two transaction identifiers are duplicated
- One amount is missing
- One amount contains non numeric text
- One amount is negative
- One timestamp represents an impossible date
- One timestamp uses a non-standard format
- Currency, transaction type and status values contain inconsistent spelling or formatting
- One transaction references a nonexistent account
- One transaction references a nonexistent merchan

<small>_No cleaning decisions have been applied yet; the next step is to define how each problem should be handled_</small>

# Data cleaning

The corrupted dataset will be cleaned using explicit rules

## Cleaning decisions:

- Exact duplicate rows will be removed
- Surrounding spaces will be removed from text values
- Currency codes will be converted to uppercase
- Transaction types and statuses will be standardised
- Numeric amounts and timestamps will be converted safely
- Valid timestamps written in a different format will be accepted
- Records with unresolved critical problems will be rejected
- Missing or unknown values will not be invented
- The original raw and dirty files will remain unchanged

In [44]:
# I start by creating a working copy of transactions_dirty

working_transactions = transactions_dirty.copy(deep = True)

In [45]:
working_transactions.shape

(62, 8)

In [46]:
# Eliminate exact duplicates

working_transactions.duplicated().sum()

np.int64(1)

In [47]:
working_transactions = (working_transactions
                        .drop_duplicates()
                        .reset_index(drop = True))

In [48]:
working_transactions.shape

(61, 8)

In [49]:
working_transactions.duplicated().sum()

np.int64(0)

In [50]:
# Normalize text columns

text_columns = [
    "transaction_id",
    "account_id",
    "merchant_id",
    "currency",
    "transaction_type",
    "status",
]


In [51]:
for column in text_columns:
    working_transactions[column] = (working_transactions[column].str.strip())

In [52]:
# Normalize currencies

working_transactions["currency"] = (working_transactions["currency"].str.upper())

In [53]:
working_transactions["currency"].value_counts(dropna = False)

currency
EUR    57
USD     3
NaN     1
Name: count, dtype: int64

In [54]:
# Normalize transaction type

working_transactions["transaction_type"] = (working_transactions["transaction_type"].str.title())

working_transactions["transaction_type"].value_counts(dropna=False)


transaction_type
Purchase    60
Refund       1
Name: count, dtype: int64

In [55]:
# Normalize status

working_transactions["status"] = (working_transactions["status"].str.title())

In [56]:
working_transactions["status"] = (working_transactions["status"]
                                  .replace(
                                      {
                                          "Complete": "Completed"
                                      }
                                  ))

In [57]:
working_transactions["status"].value_counts(dropna=False)


status
Completed    58
Declined      3
Name: count, dtype: int64

In [58]:
# Amount conversion

working_transactions["amount_parsed"] = (pd.to_numeric(working_transactions["amount"],
                                                       errors="coerce"))

In [59]:
working_transactions.loc[working_transactions["amount_parsed"].isna(), ["transaction_id", "amount", "amount_parsed"]]

,transaction_id,amount,amount_parsed
6,T0007,NaN,NaN
17,T0018,not_available,NaN


In [60]:
working_transactions.loc[
    working_transactions["amount_parsed"].notna()
    & (working_transactions["amount_parsed"] <= 0),
    [
        "transaction_id",
        "amount",
        "amount_parsed",
    ],
]

,transaction_id,amount,amount_parsed
39,T0040,-68.20,-68.2


In [61]:
# Timestamp conversion

working_transactions["timestamp_parsed"] = (pd.to_datetime(working_transactions["timestamp"],
                                                           format = "mixed",
                                                           dayfirst = True,
                                                           errors = "coerce"))

working_transactions.loc[
    working_transactions["transaction_id"].isin(
        ["T0012", "T0032"] ),
    [ "transaction_id", "timestamp", "timestamp_parsed",]
]

,transaction_id,timestamp,timestamp_parsed
11,T0012,2025-02-30 09:45:00,NaT
31,T0032,14/02/2025 21:10,2025-02-14 21:10:00


In [62]:
accounts_path = (project_root / "data" / "raw" / "accounts.csv")
merchants_path = (project_root / "data" / "raw" / "merchants.csv")

accounts = pd.read_csv(accounts_path)
merchants = pd.read_csv(merchants_path)

In [63]:
# Validation rules
# I will create a aux table, where each column will indicate if a row is failing to comply with a rule

valid_currencies = {"EUR", "USD"}
valid_transaction_types = {"Purchase", "Refund"}
valid_statuses = {"Completed", "Declined"}

duplicate_id_issue = (working_transactions["transaction_id"].duplicated(keep=False))

invalid_amount_issue = (working_transactions["amount_parsed"].isna()) | (working_transactions["amount_parsed"] <= 0)
invalid_timestamp_issue = working_transactions["timestamp_parsed"].isna()
invalid_currency_issue = working_transactions["currency"].isna() | ~working_transactions["currency"].isin(valid_currencies)
invalid_type_issue = (~working_transactions["transaction_type"].isin(valid_transaction_types))
invalid_status_issue = (~working_transactions["status"].isin(valid_statuses))
invalid_account_issue = (~working_transactions["account_id"].isin(accounts["account_id"]))
invalid_merchant_issue = (~working_transactions["merchant_id"].isin(merchants["merchant_id"]))




In [64]:
quality_issues = pd.DataFrame(
    {
        "duplicate_transaction_id": (
            duplicate_id_issue
        ),
        "invalid_amount": (
            invalid_amount_issue
        ),
        "invalid_timestamp": (
            invalid_timestamp_issue
        ),
        "invalid_currency": (
            invalid_currency_issue
        ),
        "invalid_transaction_type": (
            invalid_type_issue
        ),
        "invalid_status": (
            invalid_status_issue
        ),
        "invalid_account": (
            invalid_account_issue
        ),
        "invalid_merchant": (
            invalid_merchant_issue
        ),
    },
    index=working_transactions.index,
)

In [65]:
quality_issues.head()

,duplicate_transaction_id,invalid_amount,invalid_timestamp,invalid_currency,invalid_transaction_type,invalid_status,invalid_account,invalid_merchant
0,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False


In [66]:
issue_summary = (quality_issues.sum().sort_values(ascending = False))
issue_summary

invalid_amount              3
duplicate_transaction_id    2
invalid_timestamp           1
invalid_currency            1
invalid_merchant            1
invalid_account             1
invalid_status              0
invalid_transaction_type    0
dtype: int64

In [67]:
# Rejecting mask

reject_mask = quality_issues.any(axis = 1)

In [68]:
reject_mask.sum()

np.int64(9)

In [69]:
# Problematic rows

working_transactions.loc[
    reject_mask,
    [
        "transaction_id",
        "account_id",
        "merchant_id",
        "timestamp",
        "amount",
        "currency",
        "transaction_type",
        "status",
    ],
].sort_values("transaction_id")

,transaction_id,account_id,merchant_id,timestamp,amount,currency,transaction_type,status
6,T0007,A011,M007,2025-01-10 18:50:00,NaN,EUR,Purchase,Completed
11,T0012,A016,M012,2025-02-30 09:45:00,22.3,EUR,Purchase,Completed
17,T0018,A003,M004,2025-01-26 12:44:00,not_available,EUR,Purchase,Completed
27,T0028,A002,M014,2025-02-09 14:30:00,9.8,NaN,Purchase,Completed
39,T0040,A014,M005,2025-02-27 12:10:00,-68.20,EUR,Purchase,Completed
44,T0045,A006,M012,2025-03-06 11:05:00,26.4,EUR,Purchase,Completed
60,T0045,A006,M010,2025-03-06 11:05:00,99.99,EUR,Purchase,Completed
50,T0051,A999,M010,2025-03-15 21:25:00,11.5,EUR,Purchase,Completed
53,T0054,A013,M999,2025-03-20 12:30:00,46.95,EUR,Purchase,Completed


In [70]:
# REJECTED
# I add the columns that explains which rules were not fulfilled by each row
transactions_rejected = (
    working_transactions.loc[reject_mask]
    .copy()
)
transactions_rejected = pd.concat(
    [
        transactions_rejected,
        quality_issues.loc[reject_mask],
    ],
    axis=1,
).reset_index(drop=True)



In [71]:
# ACCEPTED
transactions_clean = (working_transactions.loc[~reject_mask]).copy()

In [72]:
print(
    f"Rows after exact duplicate removal: "
    f"{len(working_transactions)}"
)

print(
    f"Accepted rows: "
    f"{len(transactions_clean)}"
)

print(
    f"Rejected rows: "
    f"{len(transactions_rejected)}"
)

Rows after exact duplicate removal: 61
Accepted rows: 52
Rejected rows: 9


In [73]:
(
    len(transactions_clean)
    + len(transactions_rejected)
    == len(working_transactions)
)

True

In [74]:
# To prepare the clean dataframe, in the acccepted rows I substitute the original columns with the converted versions

transactions_clean["amount"] = transactions_clean["amount_parsed"] 
transactions_clean["timestamp"] = transactions_clean["timestamp_parsed"]

# I also eliminate the aux columns

transactions_clean = (transactions_clean.drop(columns = ["amount_parsed", "timestamp_parsed"]).reset_index(drop = True))

In [75]:
transactions_clean.head()

,transaction_id,account_id,merchant_id,timestamp,amount,currency,transaction_type,status
0,T0001,A001,M001,2025-03-01 08:15:00,34.80,EUR,Purchase,Completed
1,T0002,A003,M002,2025-04-01 10:42:00,4.20,EUR,Purchase,Completed
2,T0003,A005,M003,2025-05-01 19:10:00,18.99,EUR,Purchase,Completed
3,T0004,A006,M004,2025-06-01 21:35:00,249.90,EUR,Purchase,Completed
4,T0005,A008,M005,2025-08-01 12:05:00,62.40,EUR,Purchase,Completed


In [76]:
transactions_clean.dtypes

transaction_id                 str
account_id                     str
merchant_id                    str
timestamp           datetime64[us]
amount                     float64
currency                       str
transaction_type               str
status                         str
dtype: object

In [77]:
# Now I will validate the clean result

transactions_clean["transaction_id"].is_unique

True

In [78]:
transactions_clean.isna().sum()

transaction_id      0
account_id          0
merchant_id         0
timestamp           0
amount              0
currency            0
transaction_type    0
status              0
dtype: int64

In [79]:
(transactions_clean["amount"] > 0).all()

np.True_

In [80]:
transactions_clean["timestamp"].notna().all()

np.True_

In [81]:
transactions_clean["account_id"].isin(accounts["account_id"]).all()

np.True_

In [82]:
# Before and after cleaning report
cleaning_report = pd.Series(
    {
        "dirty_rows": len(
            transactions_dirty
        ),
        "exact_duplicates_removed": (
            len(transactions_dirty)
            - len(working_transactions)
        ),
        "rows_after_duplicate_removal": len(
            working_transactions
        ),
        "accepted_rows": len(
            transactions_clean
        ),
        "rejected_rows": len(
            transactions_rejected
        ),
        "accepted_missing_values": int(
            transactions_clean
            .isna()
            .sum()
            .sum()
        ),
        "accepted_ids_are_unique": (
            transactions_clean[
                "transaction_id"
            ].is_unique
        ),
    },
    name="value",
)

cleaning_report

dirty_rows                        62
exact_duplicates_removed           1
rows_after_duplicate_removal      61
accepted_rows                     52
rejected_rows                      9
accepted_missing_values            0
accepted_ids_are_unique         True
Name: value, dtype: object

In [83]:
# Save results

processed_directory = (
    project_root
    / "data"
    / "processed"
)

clean_transactions_path = (
    processed_directory
    / "transactions_clean.csv"
)

rejected_transactions_path = (
    processed_directory
    / "transactions_rejected.csv"
)




In [84]:
# Check folder existance
processed_directory.mkdir(parents = True, exist_ok = True)

In [85]:
# Save dataframes
transactions_clean.to_csv(clean_transactions_path, index = False)

In [86]:
transactions_rejected.to_csv(rejected_transactions_path, index= False)

In [87]:
print(clean_transactions_path.exists())
print(rejected_transactions_path.exists())

True
True


In [88]:
# Verify saved dataframes

saved_clean_transactions = pd.read_csv(clean_transactions_path,parse_dates=["timestamp"],
)

saved_clean_transactions.info()

<class 'pandas.DataFrame'>
RangeIndex: 52 entries, 0 to 51
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   transaction_id    52 non-null     str           
 1   account_id        52 non-null     str           
 2   merchant_id       52 non-null     str           
 3   timestamp         52 non-null     datetime64[us]
 4   amount            52 non-null     float64       
 5   currency          52 non-null     str           
 6   transaction_type  52 non-null     str           
 7   status            52 non-null     str           
dtypes: datetime64[us](1), float64(1), str(6)
memory usage: 3.4 KB


In [89]:
saved_clean_transactions.shape

(52, 8)

In [90]:
saved_rejected_transactions = pd.read_csv(
    rejected_transactions_path
)


In [91]:
saved_rejected_transactions[
    [
        "transaction_id",
        "amount",
        "timestamp",
        "duplicate_transaction_id",
        "invalid_amount",
        "invalid_timestamp",
        "invalid_currency",
        "invalid_account",
        "invalid_merchant",
    ]
]

,transaction_id,amount,timestamp,duplicate_transaction_id,invalid_amount,invalid_timestamp,invalid_currency,invalid_account,invalid_merchant
0,T0007,NaN,2025-01-10 18:50:00,False,True,False,False,False,False
1,T0012,22.3,2025-02-30 09:45:00,False,False,True,False,False,False
2,T0018,not_available,2025-01-26 12:44:00,False,True,False,False,False,False
3,T0028,9.8,2025-02-09 14:30:00,False,False,False,True,False,False
4,T0040,-68.20,2025-02-27 12:10:00,False,True,False,False,False,False
5,T0045,26.4,2025-03-06 11:05:00,True,False,False,False,False,False
6,T0051,11.5,2025-03-15 21:25:00,False,False,False,False,True,False
7,T0054,46.95,2025-03-20 12:30:00,False,False,False,False,False,True
8,T0045,99.99,2025-03-06 11:05:00,True,False,False,False,False,False


## Cleaning results

The corrupted dataset was cleaned without modifying the original raw files

The cleaning process:

- Removed one exact duplicate row
- Standardised text formatting
- Converted currency codes to uppercase
- Normalised transaction types and statuses
- Parsed valid numeric amounts and timestamps
- Accepted a valid timestamp written in a different format
- Rejected records with unresolved critical problems
- Rejected both versions of a conflicting transaction identifier
- Preserved rejected records together with their validation indicators

The resulting clean dataset contains:

- Unique transaction identifiers
- Valid and positive numeric amounts
- Valid timestamps
- Supported currencies, transaction types and statuses
- Existing account and merchant references
- No missing values